# 3. Manuscript figures and tables
Run all cells to write 13 vector PDFs for the 10 numerical figures (Fig. 7 has two panels and Fig. 8 three) and LaTeX for Tables II and IV–VIII into `build/`. The cost script also writes the earlier four-panel layout of the fixed-accuracy panels and its separate ratio figure. This notebook runs independently of notebooks 1 and 2. Tables I and III are written by hand in the manuscript, and Table VI contains analytical formulas. The finite-shot table uses the archived estimates of individual trials.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
if not (DATA / "structural_counts.tsv").is_file():
    raise FileNotFoundError("Start the notebook with its working directory at the repository root.")
sys.path.insert(0, str(ROOT / "lib"))

def read_data(filename):
    return json.loads((DATA / filename).read_text())

import hashlib
import math
import re
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
import qpe_cost_comparison as qpe_model

OUT = ROOT / "build"
OUT.mkdir(exist_ok=True)
COUNT_TSV = DATA / "structural_counts.tsv"
COUNT_SCRIPT = ROOT / "lib/counting_Hamiltonian_terms.jl"
TABLE1_TEX = OUT / "table1_counts.tex"
def save_resource_figure(filename, **kwargs):
    plt.savefig(OUT / filename, **kwargs)
def cnuc(nuc): # convert "2H" to "$^{2}$H"
    A = re.findall(r'\d+', nuc)
    element = nuc.replace(A[0], "")
    return f"$^{{{A[0]}}}${element}"

# Manuscript colors from Figure 7.
paper_red, paper_orange, paper_blue, paper_green = "#af4048", "#b6581b", "#3768a6", "#397c50"
paper_marker_alpha = 0.85
paper_line_alpha = 0.80

# Typography for the manuscript figures at their printed column widths.
PAPER_PLOT_FONT = {
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif", "pdf.fonttype": 42, "ps.fonttype": 42,
}
PAPER_PLOT_STYLE = {
    **PAPER_PLOT_FONT,
    "font.size": 11, "axes.labelsize": 12, "axes.titlesize": 11,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
}

## Fig. 3 · nonzero matrix elements
Convert recorded double-precision array sizes to element counts. Symbols distinguish the J and M schemes.

In [ ]:
plt.rcParams.update(PAPER_PLOT_STYLE)
nn_color, three_n_color = paper_blue, paper_red
fig = plt.figure(figsize=(4.65,3.4))
ax = fig.add_subplot(1,1,1)
ms = 8
lw = 1.5
df = pd.read_csv(DATA / "nonzero_me.csv", sep=r'\s*,\s*', engine='python')
_ = df[(df["scheme"]=="J") & (df["NN or 3N"]=="NN")]
ndarray_size = _['size (GB)'].values
num_array = ndarray_size * 1024**3 / 8  # number of elements
ax.plot(_['emax'], num_array, marker='o', c=nn_color, ms=ms, lw=lw, mfc='none', ls='-', label="NN, J-scheme")

_ = df[(df["scheme"]=="M") & (df["NN or 3N"]=="NN")]
ndarray_size = _['size (GB)'].values
num_array = ndarray_size * 1024**3 / 8  # number of elements
ax.plot(_['emax'], num_array, marker='o', c=nn_color, ms=ms, lw=lw, mfc=nn_color, ls='-', label="NN, M-scheme")

_ = df[(df["scheme"]=="J") & (df["NN or 3N"]=="3N")]
ndarray_size = _['size (GB)'].values
num_array = ndarray_size * 1024**3 / 8  # number of elements
ax.plot(_['emax'], num_array, marker='D', c=three_n_color, ms=ms, lw=lw, mfc='none', ls='-', label="3N, J-scheme")

_ = df[(df["scheme"]=="M") & (df["NN or 3N"]=="3N")]
ndarray_size = _['size (GB)'].values
num_array = ndarray_size * 1024**3 / 8  # number of elements

ax.plot(_['emax'], num_array, marker='D', c=three_n_color, ms=ms, lw=lw, mfc=three_n_color, ls='-', label="3N, M-scheme")

ax.legend(fontsize=10)
ax.set_yscale('log')
ax.set_xlabel(r"$e_{\rm max}$", fontsize=12)
ax.set_ylabel(r"Number of non-zero elements", fontsize=12)
ax.set_xticks(range(df['emax'].min(), df['emax'].max()+1, 2))
ax.tick_params(axis='both', labelsize=10)
fig.tight_layout()
plt.savefig(OUT / "nonzero_me.pdf", pad_inches=0.1, bbox_inches='tight',
            transparent=True)
plt.close()

## Table II and Fig. 7 · structural counts and resource formulas
The coefficient-independent counts include the identity. The universal Majorana measurement prescription enters the QKrylov estimate.

Set `FIG7_SAVE_GREEDY_COMPARISON = True` in the Fig. 7 plotting cell to also save `Tgate_counts_{valence,nocore}_greedy_majorana.pdf`. Purple triangles show QKrylov costs computed with the achieved Table V Majorana settings. Both estimates use the same structural term count for time evolution. The triangles are shown only for completed groupings, without a fit.

In [ ]:
from math import comb as integer_comb

def eval_num_Zdiag_terms(Nq, is_3NF: bool):
    Nq = int(Nq)
    return 1 + Nq + integer_comb(Nq, 2) + (integer_comb(Nq, 3) if is_3NF else 0)

def T_formula_QFT(N_ancilla, eps_rotation: float=1.e-10):
    Teps = np.ceil( 3 * np.log2(1/eps_rotation))
    # Conservative controlled-phase decomposition: three rotations.
    return N_ancilla * (N_ancilla - 1) // 2 * 3 * Teps

def T_formula_geo_series_controlled_U(N_ancilla, nterm, eps_rotation: float=1.e-10):
    T_cU_unit = 2 * (nterm - 1)
    T_eps = np.ceil( 3 * np.log2(1/eps_rotation)) 
    return T_cU_unit * (2**(N_ancilla) - 1 ) * T_eps

def T_formula_for_QPE(Na, nterm):
    T_TE = T_formula_geo_series_controlled_U(Na, nterm)
    TQFT = T_formula_QFT(Na)
    return T_TE + TQFT

def constructive_majorana_settings(nqubits, is_3nf, nterm):
    """Universal setting bound for Fig. 7; nterm includes the identity.

    The same prescription applies to every space. No empirical fit or
    selective substitution of the greedy benchmark counts is used.
    """
    from math import isqrt
    def quartic_cover(q):
        p = 2*int(q)
        while p % 4 != 3 or any(p % d == 0 for d in range(2, isqrt(p)+1)):
            p += 1
        return p*(p-1)//2
    q = int(nqubits)
    bound = quartic_cover(q)
    if is_3nf:
        size, remainder = divmod(2*q, 5)
        anchors = remainder*integer_comb(size+1, 2)+(5-remainder)*integer_comb(size, 2)
        bound += anchors*quartic_cover(q-1)
    return min(int(nterm)-1, bound)


def T_formula_for_QKrylov(Niter, nterm, Ncirc, eps_rotation: float=1.e-10):
    T_eps = np.ceil( 3 * np.log2(1/eps_rotation))
    T_U = (nterm - 1) * T_eps
    Titer = (4 * Niter**3 + Niter**2 - 3 * Niter) / 2
    return T_U * Titer * Ncirc

def T_formula_for_ODMD(Niter, nterm, eps_rotation: float=1.e-10):
    T_eps = np.ceil( 3 * np.log2(1/eps_rotation))
    T_cU = 2 * (nterm - 1) * T_eps
    Titer = Niter * (Niter + 1)
    return T_cU * Titer

def load_structural_counts(path, source_path):
    integer_columns = ["Nq", "T-estimates-controlled", "Num of Hterms", "IZterms",
                       "emin", "emax", "e2max", "threads", "Tgates_per_Rz"]
    counts = pd.read_csv(path, sep="\t", dtype={**{c: "int64" for c in integer_columns},
                                              "e3max": "Int64"})
    required = set(integer_columns) | {"Interaction-type", "modelspace", "e3max",
                                      "count_kind", "identity_included", "source_sha256"}
    if not required.issubset(counts.columns) or counts.empty:
        raise ValueError("The TSV is incomplete or uses the old schema. Regenerate it with Julia.")
    source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
    if not counts["source_sha256"].eq(source_hash).all():
        raise ValueError("The TSV is stale: regenerate it after changing the Julia code or settings.")
    if counts.duplicated(["Interaction-type", "modelspace"]).any():
        raise ValueError("Duplicate model spaces in the count table.")
    if not counts["count_kind"].eq("symmetry_structural").all() or not counts["identity_included"].eq(True).all():
        raise ValueError("Expected structural counts including the identity.")
    if not ((counts["Nq"] > 0) & (counts["IZterms"] >= 1) &
            (counts["Num of Hterms"] >= counts["IZterms"])).all():
        raise ValueError("Inconsistent term counts.")
    for _, row in counts.iterrows():
        is_3nf = row["Interaction-type"] == "NN3NF"
        if is_3nf != pd.notna(row["e3max"]):
            raise ValueError("Missing or unexpected E3max metadata.")
        if not is_3nf or row["e3max"] == 3 * row["emax"]:
            if row["IZterms"] != eval_num_Zdiag_terms(row["Nq"], is_3nf):
                raise ValueError(f"Full-space diagonal count mismatch: {row['modelspace']}")
    return counts

df = load_structural_counts(COUNT_TSV, COUNT_SCRIPT)
Na, Na_small, Niter = 20, 10, 50
for ancillas in (Na_small, Na):
    df[f"QPE (Na={ancillas})"] = [T_formula_for_QPE(ancillas, n) for n in df["Num of Hterms"]]
df["Majorana settings bound"] = [constructive_majorana_settings(row["Nq"], row["Interaction-type"] == "NN3NF", row["Num of Hterms"]) for _, row in df.iterrows()]
df["QKrylov"] = [T_formula_for_QKrylov(Niter, row["Num of Hterms"], row["Majorana settings bound"]) for _, row in df.iterrows()]
df["ODMD"] = [T_formula_for_ODMD(Niter, n) for n in df["Num of Hterms"]]
df[["Interaction-type", "modelspace", "Nq", "Num of Hterms"]].head()

In [ ]:
def write_table1(counts, output_path):
    """Generate the manuscript table from the same validated counts used in the plots."""
    if output_path is None:
        return
    def number(value):
        return "$" + f"{int(value):,}".replace(",", r"\,") + "$"

    three_body = counts[counts["Interaction-type"] == "NN3NF"]
    full_3n = bool((three_body["e3max"] == 3 * three_body["emax"]).all())
    cutoff_note = []
    if not three_body.empty:
        cutoff_note = [r"The NN+3N entries use $E_\mathrm{3max}=3e_\mathrm{max}$."
                       if full_3n else
                       r"For NN+3N entries, $E_\mathrm{3max}$ is given with each model space."]
    lines = [
        r"\begin{table}[t]", r"\centering",
        r"\caption{Model spaces, system-qubit counts $N_q$, and counts $N_{\hat{H}}$ of distinct Jordan--Wigner Pauli strings used in the resource estimates.",
        r"Counts include one-body contributions and the identity, with all proton and neutron interaction channels included.",
        r"They follow from the operator structure and symmetry selection rules; interaction-specific coefficient zeros and cancellations are not evaluated.",
        *cutoff_note,
        r"\label{tab:ModelSpace}}",
        r"\begin{ruledtabular}", r"\begin{tabular}{lrr}",
        r"Model space & $N_q$ & $N_{\hat{H}}$ \\",
    ]
    sections = [("valence NN", "Valence, NN"),
                ("NN", "No-core, NN"),
                ("NN3NF", "No-core, NN+3N")]
    for interaction, title in sections:
        group = counts[counts["Interaction-type"] == interaction]
        if group.empty:
            continue
        if interaction != "valence NN":
            group = group.sort_values(["emax", "e3max"], na_position="first")
        lines += [r"\hline", r"\multicolumn{3}{c}{\textit{" + title + r"}} \\"]
        for _, row in group.iterrows():
            if interaction == "valence NN":
                space = row["modelspace"].replace("-shell", "").replace("-", "")
                if not space.isalpha():
                    raise ValueError(f"Unexpected valence label: {space}")
                label = r"$sd$--$pf$--$sdg$" if space == "sdpfsdg" else f"${space}$ shell"
            else:
                label = f"$e_\\mathrm{{max}}={int(row['emax'])}$"
                if interaction == "NN3NF" and not full_3n:
                    cutoff = f"$E_\\mathrm{{3max}}={int(row['e3max'])}$"
                    label = r"\shortstack[l]{" + label + r"\\" + cutoff + "}"
            lines.append(f"{label} & {number(row['Nq'])} & {number(row['Num of Hterms'])} " + r"\\")
    lines += [r"\end{tabular}", r"\end{ruledtabular}", r"\end{table}", ""]
    # Replace only after the complete table has been generated.
    import os
    import tempfile
    with tempfile.NamedTemporaryFile("w", dir=output_path.parent, suffix=".tex", delete=False) as tmp:
        temporary_path = Path(tmp.name)
        try:
            tmp.write("\n".join(lines))
            tmp.close()
            os.replace(temporary_path, output_path)
        finally:
            temporary_path.unlink(missing_ok=True)
    print(f"Generated Table 1: {output_path}")

write_table1(df, TABLE1_TEX)

In [ ]:
# Figure 7 layout -- panels (a,b) at fixed algorithm parameters.
# Figure 8 uses qpe_model.SPLIT_LAYOUT. The (c,d) keys lay out the earlier
# four-panel version of the fixed-accuracy panels, which the cost script
# also writes.

import json
from matplotlib.lines import Line2D

FIG6_SIZE = (4.8, 3.55)       # width, height per panel (inches)
FIG6_LEGEND_SIZE = 10.0     # points
# Coordinates are fractions of the axes: (0, 0) is the lower-left corner.
# Label and interaction-key anchors mark the upper-left corner.
# Method-key anchors mark the lower-right corner.
FIG6_LABEL_POS = {
    "a": (0.023, 0.965), "b": (0.023, 0.965),
    "c": (0.023, 0.965), "d": (0.023, 0.965),
}
FIG6_LEGEND_POS = {
    "a": (0.982, 0.012), "b": (0.982, 0.012),
    "c": (0.978, 0.038), "d": (0.978, 0.038),
}
FIG6_FAMILY_POS = {
    "a": (0.02, 0.845), "b": (0.02, 0.845),
    "c": (0.02, 0.745), "d": (0.02, 0.745),
}
FIG6_SHOW = False            # show each saved panel in the notebook
FIG7_SAVE_GREEDY_COMPARISON = False  # True also saves both panels with Table V greedy markers.


def greedy_qkrylov_rows(counts, niter):
    """Only replace Ncirc. Use the same structural Trotter cost as the original Fig. 7."""
    operators = {row["id"]: row for row in read_data("operator_summaries.json")["rows"]}
    settings = []
    for row in read_data("grouping.json"):
        result = row["methods"].get("pairing_greedy", {})
        if result.get("status") != "computed":
            continue
        operator = operators[row["id"]]
        family = operator["family"]
        if row["nqubits"] != operator["nqubits"] or not 1 <= result["settings"] <= row["nonidentity_terms"]:
            raise ValueError(f"Inconsistent Majorana grouping for {row['id']}")
        settings.append(("valence NN" if family == "valence" else family,
                         row["nqubits"], operator.get("E3max"), result["settings"]))

    # Nq is shared across interaction families. Also match the three-body truncation.
    keys = ["Interaction-type", "Nq", "e3max"]
    measured = pd.DataFrame(settings, columns=[*keys, "Greedy Majorana settings"])
    measured["e3max"] = measured["e3max"].astype("Int64")
    rows = counts.merge(measured, on=keys, how="inner", validate="one_to_one")
    rows["QKrylov (greedy)"] = T_formula_for_QKrylov(
        niter, rows["Num of Hterms"], rows["Greedy Majorana settings"])
    return rows


colors = [paper_red, paper_orange, paper_blue, paper_green]
# Panel definitions for fixed algorithm parameters
def main_plot(Data, valence, Na, overlay_with_NERSCroadmap=False, *, greedy_rows=None):
    panel = "a" if valence else "b"
    space = "Valence" if valence else "No-core"
    if overlay_with_NERSCroadmap:
        fig, ax = plt.subplots(figsize=(9, 4.3))
        ax.set(xscale="log", xlim=(1, 1e5), ylim=(0, 19),
               xlabel="Number of system qubits $N_q$", ylabel="T-gate count")
        ax.spines[["top", "right"]].set_visible(False)
        ticks = range(0, 19, 3)
        ax.set_yticks(ticks, [rf"$10^{{{tick}}}$" for tick in ticks])
        ax.grid(True, ls="--", alpha=0.5)
        ax.set_title(f"({panel}) {space}, Trotter ($r=1$)", loc="left")
    else:
        fig, ax = qpe_model.figure6_axes(panel, f"{space}, $r=1$", figsize=FIG6_SIZE,
                                         label_anchors=FIG6_LABEL_POS)

    methods = [(f"QPE (Na={Na})", rf"QPE ($N_a={Na}$)", "s"),
               (f"QPE (Na={Na_small})", rf"QPE ($N_a={Na_small}$)", "^"),
               ("QKrylov", "QKrylov", "o"), ("ODMD", "ODMD", "D")]
    families = ["valence NN"] if valence else ["NN", "NN3NF"]
    handles = []
    fit_exponents = {family: [] for family in families}
    for i, (column, label, marker) in enumerate(methods):
        color = colors[i]
        handles.append(Line2D([], [], ls="", marker=marker, color=color,
                              markeredgecolor="none", label=label))
        for family in families:
            rows = Data.loc[Data["Interaction-type"] == family].sort_values("Nq")
            if rows.empty:
                fit_exponents[family].append(None)
                continue
            x = rows["Nq"].to_numpy()
            y = np.log10(rows[column].to_numpy())
            three_body = family == "NN3NF"
            ax.scatter(x, y, s=40, marker=marker, color=color, alpha=qpe_model.PAPER_MARKER_ALPHA,
                       edgecolors="black" if three_body else "none", linewidths=0.85)
            if len(x) < 2:
                fit_exponents[family].append(None)
                continue
            exponent, intercept = np.polyfit(np.log10(x), y, 1)
            fit_exponents[family].append(exponent)
            x_line = np.geomspace(x.min(), x.max(), 100)
            ax.plot(x_line, intercept + exponent * np.log10(x_line), color=color,
                    ls="-." if three_body else "--", lw=0.85, alpha=qpe_model.PAPER_LINE_ALPHA, zorder=1)
            if overlay_with_NERSCroadmap:
                # Direct labels belong to the optional roadmap overlay.
                dx = [1.25, 1.25, 1.18, 2.2][i]
                dy = [0.55, -0.7, 0.7, 0.5][i]
                end = -1 if valence else 0
                label_x, label_y = x[end] * dx, y[end] + dy
                ax.text(label_x, label_y, rf"$N_q^{{{exponent:.1f}}}$", fontsize=10,
                        va="center", color=color,
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.3))

    if greedy_rows is not None:
        greedy_color = "#863c83"
        for family in families:
            points = greedy_rows.loc[greedy_rows["Interaction-type"] == family]
            if points.empty:
                continue
            # Completed groupings only. No fit or extrapolation of the greedy series.
            ax.scatter(points["Nq"], np.log10(points["QKrylov (greedy)"]),
                       marker="v", s=47, color=greedy_color, zorder=3,
                       edgecolors="black" if family == "NN3NF" else "none", linewidths=0.85)
        # Reserve a legend strip so both QKrylov prescriptions are visible above the data.
        height = fig.get_figheight()
        fig.set_figheight(height + 0.31)
        scale = height / fig.get_figheight()
        fig.subplots_adjust(top=fig.subplotpars.top * scale, bottom=fig.subplotpars.bottom * scale)
        comparison_key = [
            Line2D([], [], ls="", marker="o", color=paper_blue, markeredgecolor="none",
                   label="QKrylov (pairing bound)"),
            Line2D([], [], ls="", marker="v", color=greedy_color, markeredgecolor="none",
                   label="QKrylov (greedy Majorana)"),
        ]
        fig.legend(handles=comparison_key, loc="upper center", ncol=2, frameon=False,
                   fontsize=9.5, handlelength=1, handletextpad=0.35, columnspacing=1.1)

    if overlay_with_NERSCroadmap:
        ax.legend(handles=handles, loc="lower right")
        if not valence:
            style_key = [Line2D([], [], ls=ls, marker="o", color="0.45", markeredgecolor=edge,
                                label=label) for ls, edge, label in
                         [("--", "none", "NN"), ("-.", "black", "NN+3N")]]
            fig.legend(handles=style_key, loc="lower left", bbox_to_anchor=(0.14, 0.12))
    else:
        qpe_model.figure6_legend(fig, panel, handles, anchors=FIG6_LEGEND_POS,
                                 family_anchors=FIG6_FAMILY_POS,
                                 fontsize=FIG6_LEGEND_SIZE, fit_exponents=fit_exponents)
    filename = "Tgate_counts_" + ("valence" if valence else "nocore")
    if overlay_with_NERSCroadmap:
        filename += "_NERSC"
    if greedy_rows is not None:
        filename += "_greedy_majorana"
    save_resource_figure(filename + ".pdf", transparent=True)
    if FIG6_SHOW:
        plt.show()
    plt.close(fig)
# Redraw Figure 7 -- rerun after changing the layout cell
with plt.rc_context(qpe_model.PAPER_PLOT_STYLE):
    main_plot(df, True, Na)
    main_plot(df, False, Na)
    if FIG7_SAVE_GREEDY_COMPARISON:
        greedy_rows = greedy_qkrylov_rows(df, Niter)
        main_plot(df, True, Na, greedy_rows=greedy_rows)
        main_plot(df, False, Na, greedy_rows=greedy_rows)

## Fig. 8 and Table VIII · matched-error resource estimates
Use the retained coefficients, identity shifts and commutator sums for all 20 Hamiltonians. Fig. 8 shows (a,b) the T-gate counts at 100 keV and (c) their ratios at three accuracies. Table IV (written below with the other tables) lists the inputs $M$, $\lambda'$ and $C_{\rm pair}$ without the identity; Table VIII includes the identity in the one-norm used for QSVT.

In [ ]:
# Energy and circuit settings used in the manuscript.
QPE_SETTINGS = {
    "energy_errors_MeV": [1.0, 0.1, 0.001],
    "figure6_energy_MeV": 0.1,
    "phase_estimation_fraction": 0.5,
    "circuit_synthesis_norm_budget": 0.001,
    "qrom_tradeoff_q": 1,
    "shots": 1,
}
qpe_costs = qpe_model.run(results_path=DATA / "operator_summaries.json", output_dir=OUT, settings=QPE_SETTINGS,
                          figure6_layout=dict(figsize=FIG6_SIZE, label_anchors=FIG6_LABEL_POS,
                                              legend_anchors=FIG6_LEGEND_POS, family_anchors=FIG6_FAMILY_POS,
                                              legend_fontsize=FIG6_LEGEND_SIZE))

## Fig. 6 · pairwise commutator coefficients
Fit the three model-space families to the published ranges.

In [ ]:
PLOT_QUANTITY = "coefficient"  # Or "bound" to use DELTA_T and TROTTER_STEPS.
DELTA_T = 0.01  # MeV^-1
TROTTER_STEPS = 1
X_LIMITS = (1, 10**4)
Y_LOG10_LIMITS = None  # Automatic limits for the selected quantity.
SHOW_SCALING_FITS = True
# Inclusive qubit ranges for unweighted log-log fits; None uses all available points.
FIT_RANGES = {"valence": (None, None), "NN": (None, None), "NN3NF": (None, None)}


def plot_Trotter_error(pair_commutators, delta_t=0.01, trotter_steps=1,
                       output_path=None, x_limits=(1, 10**4),
                       y_log10_limits=None, show_scaling_fits=True, fit_ranges=None,
                       quantity="coefficient"):
    """Plot C_pair by default, preserving the original notebook's figure style.

    Each record needs family ('valence', 'NN', or 'NN3NF'), nqubits, and C
    (the pairwise commutator coefficient in MeV^2). Optional main_figure=False
    excludes a record. No structural-count table or legacy log is consulted.
    Fits are unweighted least squares in log10(C) versus log10(Nq), drawn only
    over the selected data range. These finite-range fits are not asymptotic laws.
    Set quantity='bound' to plot the dimensionless B = t^2 C_pair/r instead.
    """
    import matplotlib.pyplot as plt

    if quantity not in ("coefficient", "bound"):
        raise ValueError("quantity must be 'coefficient' or 'bound'.")
    factor=1.0
    if quantity=="bound":
        if not np.isfinite(delta_t) or delta_t == 0:
            raise ValueError("A finite nonzero evolution time is required for the log plot.")
        if not isinstance(trotter_steps, (int, np.integer)) or trotter_steps < 1:
            raise ValueError("trotter_steps must be a positive integer.")
        factor=delta_t**2/trotter_steps
    if y_log10_limits is None:
        y_log10_limits=(0,15) if quantity=="coefficient" else (-4,13)
    styles = [("valence", "Valence space", "X", "#af4048"),
              ("NN", "NCSM (NN-only)", "s", "#3768a6"),
              ("NN3NF", "NCSM (NN+3N)", "^", "#397c50")]
    rows = [r for r in pair_commutators if r.get("main_figure", True)]
    fit_ranges = fit_ranges or {}
    fits = {}
    if not rows:
        raise ValueError("No pairwise commutator records were selected.")
    for row in rows:
        if row["family"] not in {s[0] for s in styles}:
            raise ValueError(f"Unknown interaction family: {row['family']}")
        if int(row["nqubits"]) != row["nqubits"] or row["nqubits"] < 1:
            raise ValueError("nqubits must be a positive integer.")
        if not np.isfinite(row["C"]) or row["C"] <= 0:
            raise ValueError("The logarithmic plot requires finite positive C values.")

    # Family colors match the cost-ratio figure in the appendix.
    with plt.rc_context({
        "font.family": "serif", "font.serif": ["DejaVu Serif"],
        "mathtext.fontset": "dejavuserif", "pdf.fonttype": 42, "ps.fonttype": 42,
        "font.size": 11, "axes.labelsize": 12, "axes.titlesize": 11,
        "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
    }):
        fig, ax = plt.subplots(figsize=(4.7, 3.35), layout="constrained")
        for family, label, marker, color in styles:
            selected = sorted((r for r in rows if r["family"] == family),
                              key=lambda r: r["nqubits"])
            if not selected:
                continue
            x = np.array([r["nqubits"] for r in selected])
            values = np.array([factor * r["C"] for r in selected])
            y = np.log10(values)
            if (np.any(x < x_limits[0]) or np.any(x > x_limits[1]) or
                    np.any(y < y_log10_limits[0]) or np.any(y > y_log10_limits[1])):
                raise ValueError("A point is outside the plot limits. Expand X_LIMITS or Y_LOG10_LIMITS.")
            # Same marker opacity and NN+3N outline as Figure 7.
            ax.scatter(x, y, color=color, marker=marker,
                       edgecolors="black" if family == "NN3NF" else "none",
                       linewidths=0.85, label=label, s=50, alpha=0.85, zorder=3)
            if show_scaling_fits:
                lower, upper = fit_ranges.get(family, (None, None))
                mask = np.ones(len(x), dtype=bool)
                if lower is not None:
                    mask &= x >= lower
                if upper is not None:
                    mask &= x <= upper
                if np.count_nonzero(mask) < 2 or len(np.unique(x[mask])) < 2:
                    raise ValueError(f"At least two distinct qubit counts are required for the {family} fit.")
                log_x = np.log10(x[mask])
                log_c = np.log10([r["C"] for r, keep in zip(selected, mask) if keep])
                exponent, intercept = np.polyfit(log_x, log_c, 1)
                residual = log_c - (intercept + exponent * log_x)
                total_variation = float(np.sum((log_c - log_c.mean())**2))
                fits[family] = dict(exponent=float(exponent), log10_prefactor_MeV2=float(intercept),
                    n_points=int(mask.sum()), min_nqubits=int(x[mask].min()), max_nqubits=int(x[mask].max()),
                    R2_log10=1-float(np.sum(residual**2))/total_variation if total_variation else None,
                    max_abs_residual_dex=float(np.max(np.abs(residual))),
                    case_ids=[r.get("id", str(r["nqubits"])) for r, keep in zip(selected, mask) if keep],
                    method="Unweighted OLS: log10(C_pair/MeV^2) = intercept + exponent*log10(Nq)")
                x_fit = np.geomspace(x[mask].min(), x[mask].max(), 200)
                y_fit = intercept + exponent*np.log10(x_fit) + np.log10(factor)
                linestyle = {"valence": ":", "NN": "--", "NN3NF": "-."}[family]
                ax.plot(x_fit, y_fit, color=color, linestyle=linestyle, lw=1.5, alpha=0.8, zorder=1)
                offset = {"valence": (7, -11), "NN": (7, 5), "NN3NF": (7, 4)}[family]
                ax.annotate(rf"$N_q^{{{exponent:.1f}}}$", (x_fit[-1], y_fit[-1]),
                            xytext=offset, textcoords="offset points", color=color, fontsize=10)
        ax.set_xscale("log")
        ax.set_xlabel(r"Number of system qubits $N_q$")
        if quantity=="coefficient":
            ax.set_ylabel(r"Pairwise coefficient $C_{\rm pair}$ (MeV$^2$)")
        else:
            ax.set_ylabel(r"Pairwise bound $B_{\rm pair}(t,r)$")
            ax.text(0.05, 0.95,
                rf"$t={delta_t:g}\,\mathrm{{MeV}}^{{-1}},\quad r={trotter_steps}$",
                transform=ax.transAxes, fontsize=14, verticalalignment="top")
        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_log10_limits)
        yticks = np.arange(y_log10_limits[0], y_log10_limits[1] + 1, 3)
        ax.set_yticks(yticks)
        ax.set_yticklabels([rf"$10^{{{tick:g}}}$" for tick in yticks])
        ax.legend(loc="lower right", fontsize=10, framealpha=0.95)
        ax.grid(True, ls="--", alpha=0.5)
        ax.set_axisbelow(True)
        fig.scaling_fits = fits
        if output_path is not None:
            fig.savefig(output_path, bbox_inches="tight", pad_inches=0.03, transparent=True)
    return fig, ax


Pair_commutators = read_data("operator_summaries.json")["rows"]
fig_bounds, ax_bounds = plot_Trotter_error(Pair_commutators, DELTA_T, TROTTER_STEPS, OUT / "trotter_error.pdf",
                                        X_LIMITS, Y_LOG10_LIMITS, SHOW_SCALING_FITS, FIT_RANGES, quantity=PLOT_QUANTITY)
plt.close(fig_bounds)

## Figs. 10 and 11 · QPE energy distributions
Recalculate both histograms from the sector Hamiltonians. The reference lines use the archived angular momenta.

In [ ]:
@plt.rc_context(PAPER_PLOT_FONT)
def plot_spectrum(n_ancilla, delta_t, target, nuc, Energy_list, Ens_exact, Jvals, save_fig=False):
    linetypes = ["solid", "dashed", "dotted", "dashdot"]
    
    if target == "NN":
        int_type = "NN-only" 
    elif target == "NN3NF":
        int_type = "NN+3NF"
    elif target == "valence":
        int_type = "Valence-space"
    else:
        int_type = ""

    fn_plot = "qpe_results_" + target + "_" + nuc + f"_Na{n_ancilla}.pdf"
    fig = plt.figure(figsize=(5,3))
    ax = fig.add_subplot(1,1,1)
    ax.set_yscale("log")
    ax.set_xlabel("Energy (MeV)")
    ax.set_ylabel("Probability")
    ax.set_ylim(bottom=1e-3, top=5.0)
    barwidth = abs(2 * np.pi / (delta_t * 2**n_ancilla) * 0.8)
    ax.bar([energy for energy, p in Energy_list], 
            [p for energy, p in Energy_list], 
            width=barwidth, color=paper_blue, alpha=paper_marker_alpha, zorder=100)

    Jhit = [ ]
    for idx, E in enumerate(Ens_exact):
        J2val = int(round(Jvals[idx]*2, 1))
        Jidx = J2val // 2
        if not (Jidx, J2val) in Jhit:
            Jhit.append((Jidx, J2val))
        linestyle = linetypes[Jidx]

        ax.axvline(E, color= "k", linestyle=linestyle, alpha=0.8, zorder=0)
    ax.text(0.03, 0.95, cnuc(nuc)+": " + int_type, transform=ax.transAxes, fontsize=14, va="top", ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="gray"))
    Emin = min(Ens_exact[0]*2, Ens_exact[0]*0.5)
    Emax = max(Ens_exact[-1]*1.5, Ens_exact[-1]*0.5)
    # Jvalue legend (original legend is reordered by J)
    legend_elements = []
    for Jidx, J2val in sorted(Jhit):
        linestyle = linetypes[Jidx]
        if J2val % 2 == 0:
            label = f"$\\mathrm{{J}}={int(J2val/2)}$"
        else:
            label = f"$\\mathrm{{J}}={J2val}/2$"
        legend_elements.append(plt.Line2D([0], [0], color="k", linestyle=linestyle, alpha=0.8, label=label))

    ax.legend(handles=legend_elements, loc="upper right", fontsize=11)
    ax.set_xlim(Emin, Emax)
    if save_fig:
        plt.savefig(OUT / fn_plot, bbox_inches='tight', pad_inches=0.05)
    plt.show()
    plt.close()

from spectra import qpe_bins
examples = read_data("demonstrations.json")
sectors = read_data("qpe_sectors.json")
for args in examples["qpe"]:
    levels, bins, probabilities = qpe_bins(sectors[args["target"]], args["delta_t"], args["n_ancilla"])
    rebuilt = np.column_stack((bins, probabilities))
    np.testing.assert_allclose(rebuilt, args["Energy_list"], atol=1e-12)
    with plt.rc_context(plt.rcParamsDefault):
        plot_spectrum(**{**args, "Energy_list": rebuilt, "Ens_exact": levels}, save_fig=True)

## Fig. 12 · QKrylov and ODMD for $^{22}$O
Plot the five lowest $J=0$ FCI energies and up to five low-lying estimates from each method. The $5\times5$ ODMD fit retains two modes.

In [ ]:
krylov = read_data("demonstrations.json")["krylov"]
evals_fci = np.asarray(krylov["evals"])
fci_j0 = np.isclose(krylov["Jvals"], 0.0, atol=1e-6)
En_history = krylov["ritz_history"]
krylov_state_count = len(En_history)
Results_ODMD = krylov["Results_ODMD"]

with plt.style.context("default"), plt.rc_context(PAPER_PLOT_FONT):
    fig, axs = plt.subplots(1, 3, figsize=(5.4, 3.0), sharey=True,
                            gridspec_kw={"width_ratios": [7, 1.5, 1.5], "wspace": 0})
    for ax in axs:
        for i, energy in enumerate(evals_fci[fci_j0][:5]):
            ax.axhline(energy, color="0.65", ls=":", lw=0.8,
                       label="FCI" if i == 0 else None)
        ax.set_ylim(evals_fci[0] - 0.8, -14.5)
        ax.tick_params(labelsize=10)
    for level in range(min(5, max(map(len, En_history)))):
        energy_history = [e[level] if level < len(e) else np.nan for e in En_history]
        axs[0].plot(range(krylov_state_count), energy_history,
                    color=paper_blue, marker="o", markersize=3, lw=0.8,
                    markeredgewidth=0, alpha=0.8,
                    label="QKrylov" if level == 0 else None)
    axs[0].set_xlabel("Iteration", fontsize=12)
    axs[0].set_ylabel("Energy (MeV)", fontsize=12)
    axs[0].set_xticks([0, 5, 10, 15])
    axs[0].set_yticks([-35, -30, -25, -20, -15])
    axs[0].text(0.04, 0.95, r"$^{22}$O", transform=axs[0].transAxes,
                fontsize=17, va="top")
    # Keep the legend inside the empty upper-right area of the QKrylov panel.
    axs[0].legend(loc="upper right", frameon=False, fontsize=10,
                  handlelength=1.8, borderaxespad=0.6)
    for ax, fit, color in zip(axs[1:], Results_ODMD, (paper_orange, paper_red)):
        low_energies = np.sort(fit["energies"])[:5]
        ax.scatter(np.zeros(len(low_energies)), low_energies,
                   color=color, marker="D", s=22, edgecolors="none", zorder=3)
        ax.set_xticks([])
        ax.tick_params(left=False)
        rows, columns = fit["shape"]
        ax.set_xlabel(f"ODMD\n{rows}×{columns}", fontsize=10)
    fig.savefig(OUT / "QKrylov_ODMD_Ens_22O.pdf", bbox_inches="tight", pad_inches=0.025)
    plt.show()
    plt.close(fig)

## Fig. 9 · angular-momentum projection
Use the saved probabilities after each complete projection cycle.

In [ ]:
projection = examples["projection"]
nuc, all_j_values = projection["nuc"], projection["all_j_values"]
projection_labels = projection["projection_labels"]
j_probability_matrix = np.array(projection["j_probability_matrix"])
np.testing.assert_allclose(j_probability_matrix.sum(axis=0), 1.0, atol=1e-10)
with plt.rc_context(PAPER_PLOT_STYLE):
    # Visualization 1: evolution of the total-J composition.
    stage_indices = np.arange(len(projection_labels))
    fig, ax = plt.subplots(
        figsize=(4.7, 3.25),
        constrained_layout=True,
    )
    spin_colors = [paper_blue, paper_orange, paper_red, paper_green, "0.45"]
    group_width = 0.5
    bar_width = group_width / len(all_j_values)
    probability_floor = 1e-8


    for row, j_value in enumerate(all_j_values):
        color = spin_colors[row % len(spin_colors)]
        probabilities = j_probability_matrix[row]
        offset = (row - (len(all_j_values) - 1) / 2) * bar_width
        visible = probabilities > probability_floor
        ax.bar(
            stage_indices[visible] + offset,
            probabilities[visible] - probability_floor,
            bottom=probability_floor,
            width=0.92 * bar_width,
            color=color, alpha=paper_marker_alpha,
            edgecolor="w",
            linewidth=0.35,
            label=rf"$J={j_value:g}$",
        )

    ax.set_xticks(stage_indices, ["Initial"] + [str(i) for i in range(1, len(projection_labels))])
    ax.set_xlabel("Projection cycle")
    ax.set_yscale("log")
    ax.set_ylim(probability_floor, 1.0)
    ax.set_ylabel(r"Angular-momentum fraction $P(J)$")

    # Leave room after the last measured cycle for the spin key.
    left, right = ax.get_xlim()
    ax.set_xlim(left, right + 0.16 * (right - left))
    ax.legend(loc="upper right", frameon=False, fontsize=10, handlelength=1.0,
              handletextpad=0.35, labelspacing=0.3, borderaxespad=0.35)
    plt.savefig(OUT / f"J_weight_history_{nuc}.pdf", bbox_inches="tight", pad_inches=0.02)
    plt.show()
    plt.close(fig)

## Figs. 14 and 13 · VQE convergence and product-formula errors
For VQE (Fig. 14) the upper panels use FCI references and the lower panels use errors relative to DOCI. Fig. 13 uses the product-formula benchmark checked independently in notebook 2.

In [ ]:
Dict_Egs_pUCCD = examples["vqe"]["history"]
Egs_FCI_dict = examples["vqe"]["fci"]
def latex_nuc(nuc: str) -> str:
    """Convert nuclear notation to LaTeX format.
    
    Args:
        nuc (str): Nuclear notation string (e.g., "6He", "12C").
        
    Returns:
        str: LaTeX formatted string with mass number as superscript.
        
    Example:
        >>> latex_nuc("6He")
        '${}^{6}$He'
    """
    Anum = re.findall(r"\d+", nuc)[0]
    return "${}^{" + Anum + "}$" + nuc.replace(Anum, "")
with plt.rc_context(PAPER_PLOT_STYLE):
    from matplotlib.ticker import MaxNLocator

    fig, axes = plt.subplots(2, 4, figsize=(9.2, 2.55), sharex="col",
                             layout="constrained")
    fig.get_layout_engine().set(w_pad=0.045, h_pad=0.025, hspace=0.03)
    for column, nuc in enumerate(["6He", "18O", "42Ca", "58Ni"]):
        energy_ax, error_ax = axes[:, column]
        history = Dict_Egs_pUCCD[nuc]
        reference = Egs_FCI_dict[nuc]

        energy_ax.plot(history["x"], history["y"], color=paper_red, alpha=paper_line_alpha, label="VQE")
        energy_ax.axhline(reference, color=paper_green, alpha=paper_line_alpha, ls="--", label="FCI")
        energy_ax.set_ylim(bottom=reference - 0.2 * abs(reference - np.max(history["y"])))
        energy_ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
        energy_ax.set_title(latex_nuc(nuc), loc="left", pad=2)
        if column == 0:
            energy_ax.legend(loc="upper right", fontsize=9, frameon=False,
                             handlelength=1.8, labelspacing=0.12, borderaxespad=0.3)

        error_ax.plot(history["x"], history["ydiff"], color=paper_red, alpha=paper_line_alpha)
        error_ax.set_yscale("log")
        error_ax.set_ylim(1e-15, 1e2)
        error_ax.set_yticks([1e-14, 1e-7, 1])
        error_ax.set_xticks([0, 200, 400])
        error_ax.tick_params(axis="x", top=True)
        if column:
            error_ax.tick_params(axis="y", labelleft=False)

    axes[0, 0].set_ylabel("Energy (MeV)", fontsize=11)
    axes[1, 0].set_ylabel(r"$\Delta E$ (MeV)", fontsize=11)
    fig.align_ylabels(axes[:, 0])
    fig.supxlabel("Iteration", fontsize=11)
    fig.savefig(OUT / "VQE_pUCCD.pdf", bbox_inches="tight", pad_inches=0.025)
    plt.show()
    plt.close(fig)

In [ ]:
from trotter_plot import plot as plot_product_errors
benchmark = read_data("trotter_order.json")
with plt.rc_context(plt.rcParamsDefault):
    plot_product_errors(benchmark["cases"], benchmark["settings"], OUT / "trotter_order_benchmark.pdf")

## Tables IV–VII
List the retained term counts, coefficient norms and commutator coefficients of the 20 Hamiltonians (Table IV). Compute the measurement-setting reduction factors from the retained-term and setting counts (Table V); a dash denotes an unfinished calculation. Table VI contains the analytical formulas. For Table VII, count the successful estimates directly from the archived trials.

In [ ]:
layouts = read_data("table_layouts.json")
def write_rows(filename, key, rows):
    line_end = r" \\"
    body = [" & ".join(row) + line_end for row in rows]
    (OUT / filename).write_text("\n".join([layouts[key]["header"], *body, layouts[key]["footer"], ""]))

group_rows = []
for case in read_data("grouping.json"):
    settings = [case["methods"].get(name, {}).get("settings")
                for name in ("qwc", "fc", "pairing_greedy")]
    if not any(settings):
        continue
    terms = case["nonidentity_terms"]
    factors = [f"{terms/n:.1f}" if n else "---" for n in settings]
    group_rows.append([layouts["grouping"]["labels"][case["id"]], str(case["nqubits"]), f"{terms:,}", *factors])
write_rows("table_grouping.tex", "grouping", group_rows)
(OUT / "table_cost_formulas.tex").write_text(layouts["formulas"] + "\n")

# Table IV: inputs of the QPE comparison at fixed energy accuracy.
def format_norm(value):
    """One decimal below 1e3, an integer below 1e4, three digits above."""
    if value < 1e3:
        return f"{value:.1f}"
    if value < 1e4:
        return f"{value:,.0f}"
    mantissa, exponent = f"{value:.2e}".split("e")
    return rf"${mantissa}\times10^{{{int(exponent)}}}$"

coefficient_layout = layouts["coefficients"]
coefficient_lines = [coefficient_layout["header"]]
previous_family = None
for row in read_data("operator_summaries.json")["rows"]:
    if not row.get("main_figure", True):
        continue
    family = row["family"]
    if family != previous_family:
        if previous_family is not None:
            coefficient_lines.append(r"\hline")
        coefficient_lines.append(r"\multicolumn{5}{c}{\textit{"
                                 + coefficient_layout["sections"][family] + r"}} \\")
        previous_family = family
    nonidentity = int(row["terms_including_identity"]) - 1
    coefficient_lines.append(" & ".join([
        coefficient_layout["labels"][row["id"]], str(row["nqubits"]), f"{nonidentity:,}",
        format_norm(row["lambda_nonidentity"]), f"{math.log10(row['C']):.2f}"]) + r" \\")
coefficient_lines += [coefficient_layout["footer"], ""]
(OUT / "table_coefficients.tex").write_text("\n".join(coefficient_lines))

In [ ]:
from scipy.stats import beta

protocols = read_data("finite_shot.json")
shot_rows = []
with np.load(DATA / "finite_shot_energies.npz", allow_pickle=False) as samples:
    for nucleus, case in protocols["cases"].items():
        baseline = next(row["W"] for row in case["selected"] if row["method"] == "QPE")
        for i, row in enumerate(case["selected"]):
            method = row["method"]
            estimates = samples[f"{nucleus}_{method}"]
            hits = int(np.count_nonzero(np.abs(estimates - case["E0"]) <= protocols["tolerance_MeV"]))
            assert len(estimates) == row["trials"] == protocols["trials_per_protocol"]
            assert hits == row["hits"]
            lower = beta.ppf(.05/6, hits, len(estimates) - hits + 1)
            np.testing.assert_allclose(lower, row["lower"], rtol=1e-12)
            shots = row["settings"]*row["n"]
            assert shots == row["shots"]
            ratio = row["W"]/baseline
            np.testing.assert_allclose(ratio, row["W_over_QPE"])
            formatted = f"{ratio:.3g}"
            if "e+" in formatted:
                mantissa, exponent = formatted.split("e+")
                formatted = rf"${mantissa}\times10^{int(exponent)}$"
            label = re.sub(r"(\d+)([A-Za-z]+)", r"$^{\1}$\2", nucleus) if i == 0 else ""
            dt = f"{row['dt']:.5f}" if row["dt"] < 0.1 and round(row["dt"], 3) != round(row["dt"], 5) else f"{row['dt']:.3f}"
            shot_rows.append([label, method, str(row["size"]), dt, f"{row['settings']:,}", f"{row['n']:,}",
                              f"{shots:,}", f"{row['max_slices']:,}", formatted, f"{hits:,}"])
write_rows("table_finite_shots.tex", "finite_shot", shot_rows)
pd.DataFrame(shot_rows, columns=["nucleus", "method", "size", "dt", "settings", "shots/setting",
                                 "total shots", "max substeps", "relative work", "successes"])